kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import anndata as ad
import pandas as pd
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "zheng_GEX"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

zheng_adata = ad.read_h5ad(os.path.join(clustered_h5ad_dir, SERIES + ".h5ad"))

In [9]:
cell_marker_genes = {
    "OPC": ["Pdgfra", "Megf11", "Vcan", "Myt1", "Has2", "Col9a1", ],
    "COP": ["Gpr17", "Bmp4", "Tcf7l2", "Bcas1", ],
    "Oligodendrocyte": ["Mbp", "Mog", "Mag", "Cnp", "Mobp", "Fa2h",
                        "Ppp1r14a", "Ermn", "Gpr37", "Myrf", ],
    "Neuron": ["Stmn2", "Rbfox3", "Eno2", "Syt1", "Cck", "Grin1", "Tmem130", ],
    "Microglia": ["Tmem119", "Siglech", "Olfml3", "Gpr34", "Vsir" ],
    "Lymphocyte": ["Cd3d", "Cd3e", "Cd3g", "Trac", "Cd2", "Cd5", "Lck", "Cd79a", "Cd79b", "Ms4a1"],
    "BAM": ["Mrc1", "Cd163", "Lyve1", "Pf4", "Ms4a7", "Stab1", "Cbr2", ], 
    "Macrophage": [ "Ly6c2", "Lgals3", "S100a11", "Ccl5", "Ccr2", "Tgfbi"],
    "Endothelial": ["Cldn5", "Cd34", "Flt1", "Vwf", "Cdh5", "Clec14a", "Erg", "Itm2a", ],
    "Pericyte": ["Rgs5", "Cd248", "Kcnj8", ],
    "Astrocyte": ["Aqp4", "Aldh1l1", "Atp1b2", "Aldh1a1", "Agt", "Atp13a4", "Bmpr1b", "Rfx4", "Slc4a4", ],
    "Ependymal": ["Foxj1", "Dynlrb2", "Dnah11", "Pifo", ],
    "VLMC": ["Col1a2", "Col3a1", ],
}

plot.plot_dotplot(zheng_adata, SERIES, cell_marker_genes, group = "leiden")

In [10]:
leiden_to_celltype = {
    "0": "Oligodendrocyte",
    "1": "Oligodendrocyte",
    "2": "Oligodendrocyte",
    "3": "Oligodendrocyte",
    "4": "Oligodendrocyte",
    "5": "Oligodendrocyte",
    "6": "Oligodendrocyte",
    "7": "Oligodendrocyte",
    "8": "Oligodendrocyte",
    "9": "Oligodendrocyte",
    "10": "Oligodendrocyte",
    "11": "Oligodendrocyte",
    "12": "Oligodendrocyte",
    "13": "Oligodendrocyte",
    "14": "Oligodendrocyte",
    "15": "Oligodendrocyte",
    "16": "Oligodendrocyte",
    "17": "Oligodendrocyte",
    "18": "Oligodendrocyte",
    "19": "Oligodendrocyte",
    "20": "Oligodendrocyte",
    "21": "Oligodendrocyte",
    "22": "Oligodendrocyte",
    "23": "Oligodendrocyte",
    "24": "Oligodendrocyte",
    "25": "Oligodendrocyte",
    "26": "Oligodendrocyte",
    "27": "Oligodendrocyte",
    "28": "Oligodendrocyte",
    "29": "Oligodendrocyte",
    "30": "Oligodendrocyte",
    "31": "Oligodendrocyte",
    "32": "Oligodendrocyte",
    "33": "Oligodendrocyte",
    "34": "Oligodendrocyte",
    "35": "Oligodendrocyte",
    "36": "Oligodendrocyte",
    "37": "Oligodendrocyte",
    "38": "Oligodendrocyte",
    "39": "Oligodendrocyte",
    "40": "Oligodendrocyte",
    "41": "Oligodendrocyte",
    "42": "Ependymal",
    "43": "VLMC",
    "44": "COP",
    "45": "Macrophage",
    "46": "OPC",
    "47": "Neuron",
    "48": "Astrocyte",
    "49": "Astrocyte",
    "50": "Lymphocyte",
    "51": "Astrocyte",
    "52": "Microglia",
    "53": "Endothelial+Pericyte",
}

leiden_str = zheng_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
zheng_adata.obs["cell"] = mapped

In [11]:
zheng_adata.obs["celltype"] = zheng_adata.obs["cell"].copy() #type: ignore
plot.plot_umap(zheng_adata, SERIES, has_celltype=True)

In [ ]:
assert isinstance(zheng_adata.obs, pd.DataFrame)
zheng_adata.obs["cell"] = zheng_adata.obs["cell"]
zheng_adata.obs["sample"] = zheng_adata.obs["sample"]
zheng_adata.obs["stage"] = zheng_adata.obs["sample"].str.split('_').str[0]
zheng_adata.obs["eae"] = zheng_adata.obs["sample"].str.split('_').str[1]
zheng_adata.obs["donor"] = zheng_adata.obs["sample"]
zheng_adata.obs["series"] = "Zheng"

keep = ["cell", "sample", "stage", "eae", "donor", "series"]
zheng_adata.obs.drop(columns = [c for c in zheng_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [22]:
zheng_adata.write_h5ad(os.path.join(final_h5ad_dir, SERIES + "_final.h5ad"))

oligo = zheng_adata[
    (zheng_adata.obs["cell"] == "OPC") |
    (zheng_adata.obs["cell"] == "COP") | 
    (zheng_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))

microglia = zheng_adata[
    (zheng_adata.obs["cell"] == "Microglia")
]
microglia.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_microglia_raw.h5ad"))